# Bài 11 — Orchestrator

## Lý thuyết

### Vấn đề cần giải quyết

Ở Bài 10, ta đã có 4 agent độc lập (Web Research, Financial Data, RAG, Analysis) nhưng phải **gọi tay từng agent** trong cell test, và luôn chạy **đủ cả 3 agent dữ liệu** (Web, Financial, RAG) dù câu hỏi có cần hay không.

**Orchestrator** là một node đặc biệt: nó **đọc câu hỏi người dùng và quyết định agent nào cần chạy**, rồi điều phối luồng xử lý — thay vì con người quyết định bằng tay như ở Bài 10.

### Khái niệm mới: Conditional Edges

Ở Bài 9, mọi `add_edge(a, b)` là **cố định**: chạy xong `a` thì luôn luôn đến `b`, không có lựa chọn nào khác. Đây gọi là **edge tĩnh**.

Nhưng Orchestrator cần **quyết định động** (dynamic): tuỳ vào nội dung câu hỏi (lưu trong state) mà đi đến node khác nhau. Đây gọi là **conditional edge**.

| Hàm/khái niệm | Ý nghĩa |
|---|---|
| `add_conditional_edges(source, routing_func, path_map=None)` | Sau khi chạy xong node `source`, gọi `routing_func(state)` để quyết định node tiếp theo, thay vì đi cố định |
| `routing_func(state)` | Hàm Python bạn viết, nhận `state`, **trả về tên node tiếp theo** (1 string), hoặc **trả về 1 list string** nếu muốn rẽ tới nhiều node cùng lúc (fan-out) |
| `path_map` (tuỳ chọn) | dict map từ giá trị `routing_func` trả về sang tên node thật, ví dụ `{"high": "high_price_node", "low": "low_price_node"}`. Nếu `routing_func` trả về đúng tên node luôn thì không cần `path_map` |

> So sánh: `add_edge` giống 1 đường ống cố định. `add_conditional_edges` giống 1 cái **biển báo rẽ nhánh** — đi hướng nào tuỳ vào "tình huống" (state) lúc đó.

**Fan-out (rẽ nhiều nhánh cùng lúc):** nếu `routing_func` trả về list, ví dụ `["financial_data_node", "rag_node"]`, LangGraph sẽ chạy **cả 2 node đó** (bỏ qua node còn lại không được chọn) — đây chính là cách Orchestrator "chọn agent nào cần chạy" một cách thật.

### Kiến trúc Orchestrator cho hệ thống 4 agent

```
                    ┌──────────────────┐
                    │   router_node    │   ← đọc câu hỏi, quyết định
                    │ (Orchestrator)   │     agent nào cần chạy
                    └─────────┬────────┘
                              │ (conditional edge — rẽ nhánh động)
              ┌───────────────┼───────────────┐
              ▼               ▼               ▼
      ┌───────────────┐┌────────────┐┌───────────┐
      │ Web Research  ││ Financial  ││    RAG    │   ← chỉ những node
      │     node      ││ Data node  ││   node    │     được chọn mới
      └───────┬───────┘└─────┬──────┘└─────┬─────┘     chạy
              └──────────────┴─────────────┘
                             ▼
                    ┌──────────────────┐
                    │  analysis_node   │   ← luôn chạy, tổng hợp
                    │                  │     những gì có sẵn trong
                    └─────────┬────────┘     state
                              ▼
                             END
```

Khác với Bài 10 (bạn gọi tay từng hàm), giờ **router_node** đóng vai trò "người quyết định" — và 3 node dữ liệu có thể **không chạy hết cả 3** tuỳ câu hỏi.

## Ví dụ — Conditional edge đơn giản

Trước khi áp dụng vào 4 agent thật (phức tạp hơn), ta làm ví dụ nhỏ để hiểu **cơ chế** `add_conditional_edges`: graph nhận giá cổ phiếu, **rẽ nhánh** theo giá cao/thấp, mỗi nhánh xử lý khác nhau, rồi **hội tụ** lại ở node in kết quả.

```
START → check_price_node → (rẽ nhánh tuỳ giá)
                              ├── high_price_node ──┐
                              └── low_price_node ───┤
                                                    ▼
                                              print_node → END
```

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict


class PriceState(TypedDict):
    price: float
    message: str


def check_price_node(state: PriceState):
    print(f"[Check] Giá nhận được: {state['price']}")
    return {}


def high_price_node(state: PriceState):
    return {"message": "Giá đang ở mức CAO"}


def low_price_node(state: PriceState):
    return {"message": "Giá đang ở mức THẤP"}


def print_node(state: PriceState):
    print(f"[Print] {state['message']}")
    return {}


# routing_func: nhận state, trả về TÊN NODE tiếp theo (string)
def route_by_price(state: PriceState):
    if state["price"] > 100:
        return "high_price_node"
    else:
        return "low_price_node"

In [2]:
graph_builder = StateGraph(PriceState)

graph_builder.add_node("check_price", check_price_node)
graph_builder.add_node("high_price_node", high_price_node)
graph_builder.add_node("low_price_node", low_price_node)
graph_builder.add_node("print_result", print_node)

graph_builder.add_edge(START, "check_price")

# add_conditional_edges: sau "check_price", gọi route_by_price(state) để biết đi đâu tiếp
graph_builder.add_conditional_edges("check_price", route_by_price)

# Cả 2 nhánh đều hội tụ về "print_result"
graph_builder.add_edge("high_price_node", "print_result")
graph_builder.add_edge("low_price_node", "print_result")
graph_builder.add_edge("print_result", END)

graph = graph_builder.compile()

In [3]:
final_state = graph.invoke({"price": 207.0, "message": ""})
print("State cuối cùng:", final_state)

final_state2 = graph.invoke({"price": 50.0, "message": ""})
print("State cuối cùng:", final_state2)

[Check] Giá nhận được: 207.0
[Print] Giá đang ở mức CAO
State cuối cùng: {'price': 207.0, 'message': 'Giá đang ở mức CAO'}
[Check] Giá nhận được: 50.0
[Print] Giá đang ở mức THẤP
State cuối cùng: {'price': 50.0, 'message': 'Giá đang ở mức THẤP'}


## Bài tập

Ghép 4 agent ở Bài 10 vào **1 LangGraph có Orchestrator**, để hệ thống tự quyết định agent nào cần chạy theo câu hỏi.

1. **Copy lại 4 hàm agent** từ Bài 10 vào notebook này: `web_research_agent`, `financial_data_agent`, `rag_agent`, `analysis_agent` (giữ nguyên logic, chỉnh lại đường dẫn ChromaDB nếu cần — notebook này cũng nằm trong `phase3/`, giống lỗi đường dẫn đã gặp ở Bài 10).

2. Định nghĩa `OrchestratorState` (`TypedDict`) gồm các field:
   - `question` (câu hỏi gốc), `symbol` (mã cổ phiếu)
   - `needed_agents`: list tên node cần chạy (do `router_node` quyết định)
   - `web_result`, `financial_result`, `rag_result`: kết quả từng agent (khởi tạo `""`)
   - `final_result`: kết quả cuối từ `analysis_agent`

3. Viết **`router_node(state)`**: đọc `state["question"]`, dùng keyword đơn giản để quyết định cần agent nào, ví dụ:
   - có "tin tức"/"news" → cần `"web_research_node"`
   - có "giá"/"price"/"vốn hóa"/"P/E" → cần `"financial_data_node"`
   - có "báo cáo"/"doanh thu"/"revenue" → cần `"rag_node"`
   - nếu câu hỏi không khớp keyword nào, mặc định chạy hết cả 3 (an toàn hơn bỏ sót)
   Trả về `{"needed_agents": [...]}`.

4. Viết **`route_to_agents(state)`** — hàm dùng cho `add_conditional_edges`: chỉ cần trả về `state["needed_agents"]` (vì các giá trị trong list đã đúng là tên node thật, không cần `path_map`).

5. Viết **3 node wrapper**, mỗi node gọi 1 hàm agent và lưu kết quả vào field tương ứng:
   - `web_research_node(state)` → gọi `web_research_agent(state["question"])`, trả `{"web_result": ...}`
   - `financial_data_node(state)` → gọi `financial_data_agent(state["symbol"])`, trả `{"financial_result": ...}`
   - `rag_node(state)` → gọi `rag_agent(state["question"])`, trả `{"rag_result": ...}`

6. Viết **`analysis_node(state)`**: chỉ gom những field **thực sự có dữ liệu** (không phải lần nào cũng có đủ cả 3!) thành 1 đoạn text, gọi `analysis_agent(...)`, trả `{"final_result": ...}`.

7. Build graph:
   - `START → router_node`
   - `add_conditional_edges("router_node", route_to_agents)`
   - Mỗi node trong `{web_research_node, financial_data_node, rag_node}` → `analysis_node`
   - `analysis_node → END`

8. Test với **ít nhất 2 câu hỏi khác nhau** để chứng minh router chọn agent khác nhau, ví dụ:
   - `"Giá NVDA hiện tại bao nhiêu?"` → chỉ nên thấy `financial_data_node` chạy
   - `"Doanh thu NVIDIA quý 1 2027 là bao nhiêu, có tin tức gì mới không?"` → nên thấy `rag_node` + `web_research_node` chạy

> Gợi ý: thêm `print(f"[node_name] đang chạy...")` ở đầu mỗi node để khi test biết chắc node nào thực sự được gọi.

Viết code của bạn vào các ô dưới đây:

In [2]:
import os
from dotenv import load_dotenv
from google import genai
from ddgs import DDGS


load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

In [ ]:
# 1. Copy 4 hàm agent từ Bài 10 (import + định nghĩa lại)

# 1. web_research_agent
def web_research_agent(query):
    # Bước 1: Tìm kiếm tin tức thật
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=3)

    # Bước 2: Ghép các kết quả tìm được thành context
    news_context = "\n\n".join([f"{r['title']}: {r['body']}" for r in results])

    # Bước 3: Đưa cho LLM tổng hợp thành câu trả lời tự nhiên
    prompt = f"""Dựa vào các tin tức sau:

{news_context}

Hãy tóm tắt thông tin quan trọng nhất."""

    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt
    )
    return response.text
import yfinance as yf

# 2. financial_data_agent
def financial_data_agent(symbol):
    ticker_obj = yf.Ticker(symbol)
    info = ticker_obj.info
    return {
        "symbol": symbol,
        "currentPrice": info.get("currentPrice", "Không tìm thấy giá cổ phiếu"),
        "marketCap": info.get("marketCap", "Không tìm thấy marketCap"),
        "trailingPE": info.get("trailingPE", "Không tìm thấy trailingPE")
    }

# 3. rag_agent
from sentence_transformers import SentenceTransformer
import chromadb

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
db_client = chromadb.PersistentClient(path=r"D:\Code\AI\Multi-Agent Research Assistant project\phase2\chroma_db")
collection = db_client.get_or_create_collection(name="nvidia_report")

def rag_agent(question):
    question_embedding = embed_model.encode(question)
    results = collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=3
    )
    chunks = results["documents"][0]
    context = "\n\n".join(chunks)

    prompt = f"""Dựa vào thông tin sau từ báo cáo tài chính NVIDIA:

{context}

Hãy trả lời câu hỏi: {question}
Nếu thông tin không đủ để trả lời, hãy nói rõ là không tìm thấy thông tin liên quan.
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt
    )
    return response.text


# 4. analysis_agent
def analysis_agent(data):
    prompt = f"""Dựa vào thông tin tổng hợp sau về tài chính NVIDIA:

{data}

Hãy tổng hợp thông tin, phân tích xu hướng. 
Chỉ sử dụng dữ liệu được cung cấp ở trên. Không tự thêm số liệu, sự kiện hay bất kì thông tin nào không có trong dữ liệu này.
    """
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=prompt
    )
    return response.text

In [ ]:
# 2. Định nghĩa OrchestratorState
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List

class OrchestratorState(TypedDict):
    question: str
    symbol: str
    needed_agents: List[str]  # Ví dụ ["financial_data", "rag"]
    web_result: str
    financial_result: dict
    rag_result: str
    final_result: str

In [31]:
# 3. router_node + 4. route_to_agents
def router_node(state: OrchestratorState):
    question = state['question'].lower()
    
    agents = []

    if "tin tức" in question or "news" in question:
        agents.append("web_research")
    
    if "giá" in question or "price" in question:
        agents.append("financial_data")
    
    if "báo cáo" in question or "doanh thu" in question or "report" in question or "revenue" in question:
        agents.append("rag")

    if not agents:
        agents = ["web_research", "financial_data", "rag"]

    print(f"[Router] Agent cần chạy: {agents}")

    return {"needed_agents": agents}


def route_to_agents(state: OrchestratorState):
    return state["needed_agents"]

In [ ]:
# 5. 3 node wrapper: web_research_node, financial_data_node, rag_node
def web_research_node(state: OrchestratorState):
    print("[Web Research Node] đang chạy...")

    result = web_research_agent(state['question'])

    return {"web_result": result}

def financial_data_node(state: OrchestratorState):
    print("[Financial Data Node] đang chạy...")

    result = financial_data_agent(state['symbol'])

    return {"financial_result": result}

def rag_node(state: OrchestratorState):
    print("[Rag Node] đang chạy...")

    result = rag_agent(state['question'])

    return {"rag_result": result}

In [33]:
# 6. analysis_node
def analysis_node(state: OrchestratorState):
    parts = []
    
    if state["web_result"]:
        parts.append(state["web_result"])
    
    if state["financial_result"]:
        parts.append(str(state["financial_result"]))

    if state["rag_result"]:
        parts.append(state["rag_result"])

    combined_data = "\n\n".join(parts)

    analysis_result = analysis_agent(combined_data)
    print("\nAnalysis Agent: ")
    return {"final_result": analysis_result}

In [34]:
# 7. Build graph (StateGraph, add_node, add_conditional_edges, add_edge, compile)
graph_builder = StateGraph(OrchestratorState)

graph_builder.add_node("web_research", web_research_node)
graph_builder.add_node("financial_data", financial_data_node)
graph_builder.add_node("rag", rag_node)
graph_builder.add_node("router", router_node)
graph_builder.add_node("analysis", analysis_node)


graph_builder.add_edge(START, "router")
graph_builder.add_conditional_edges("router", route_to_agents)

graph_builder.add_edge("web_research", "analysis")
graph_builder.add_edge("financial_data", "analysis")
graph_builder.add_edge("rag", "analysis")
graph_builder.add_edge("analysis", END)

graph = graph_builder.compile()

In [10]:
# 8. Test với ít nhất 2 câu hỏi khác nhau (graph.invoke)
result1 = graph.invoke({
    "question": "Giá NVDA hiện tại bao nhiêu?",
    "symbol": "NVDA",
    "needed_agents": [],
    "web_result": "",
    "financial_result": {},
    "rag_result": "",
    "final_result": ""
})
print(result1["final_result"])

[Router] Agent cần chạy: ['financial_data']
[Financial Data Node] đang chạy...

Analysis Agent: 
Dựa trên thông tin tổng hợp được cung cấp:

**Thông tin tổng hợp về NVIDIA (NVDA):**

*   **Mã chứng khoán:** NVDA
*   **Giá cổ phiếu hiện tại:** $200.75
*   **Vốn hóa thị trường:** $4,862,365,925,376
*   **Chỉ số P/E trailing:** 30.789877 (làm tròn thành 30.79)

**Phân tích xu hướng (chỉ dựa trên dữ liệu hiện có):**

Với dữ liệu được cung cấp chỉ là một ảnh chụp tại một thời điểm duy nhất, chúng ta không thể xác định xu hướng tăng hay giảm của giá cổ phiếu hoặc các chỉ số khác theo thời gian vì không có dữ liệu lịch sử. Tuy nhiên, chúng ta có thể phân tích các đặc điểm mà những số liệu này thể hiện ở thời điểm hiện tại:

1.  **Vốn hóa thị trường:** Mức vốn hóa $4,862,365,925,376 (khoảng 4.86 nghìn tỷ USD) cho thấy NVIDIA là một công ty có quy mô cực kỳ lớn, là một trong những doanh nghiệp có giá trị nhất trên thị trường. Điều này phản ánh vị thế và tầm ảnh hưởng đáng kể của công ty trong n

In [16]:
result2 = graph.invoke({
    "question": "Doanh thu NVIDIA quý 1 2027 là bao nhiêu, có tin tức gì mới không?",
    "symbol": "NVDA",
    "needed_agents": [],
    "web_result": "",
    "financial_result": {},
    "rag_result": "",
    "final_result": ""
})
print(result2["final_result"])

[Router] Agent cần chạy: ['web_research', 'rag']
[Rag Node] đang chạy...
[Web Research Node] đang chạy...

Analysis Agent: 
Dựa trên thông tin được cung cấp, dưới đây là tổng hợp và phân tích xu hướng về tài chính của NVIDIA:

---

**Tổng hợp thông tin & Phân tích xu hướng tài chính NVIDIA**

**I. Tổng quan Tài chính Nổi bật & Xu hướng Tăng trưởng:**

*   **Doanh thu kỷ lục:** NVIDIA đã công bố doanh thu quý kỷ lục đạt **81,6 tỷ USD** trong Quý 1 năm tài chính 2027 (kết thúc ngày 26 tháng 4 năm 2026), vượt xa dự báo của Phố Wall.
    *   **Xu hướng:** Điều này khẳng định đà tăng trưởng mạnh mẽ và khả năng vượt kỳ vọng thị trường của công ty.
*   **Tăng trưởng ấn tượng:** Doanh thu quý 1 FY27 tăng **20%** so với quý trước và tăng **85%** so với cùng kỳ năm trước.
    *   **Xu hướng:** Mức tăng trưởng cao liên tục cho thấy nhu cầu thị trường đối với sản phẩm và dịch vụ của NVIDIA vẫn đang bùng nổ.
*   **Động lực từ Trung tâm dữ liệu (Data Center):** Doanh thu từ mảng Data Center đạt kỷ l

In [ ]:
# Test nhánh mặc định: câu hỏi không khớp keyword nào -> chạy cả 3 agent
result3 = graph.invoke({
    "question": "Bạn nhận định chung thế nào về NVIDIA?",
    "symbol": "NVDA",
    "needed_agents": [],
    "web_result": "",
    "financial_result": {},
    "rag_result": "",
    "final_result": ""
})
print(result3["final_result"])